# Data Processing

In [ ]:
# ==========================================
# SETUP AND STRICT SPLITTING
# ==========================================

import os
import shutil
import random
import torch
import torchaudio
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import kagglehub
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import numpy as np
import time
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR

# Set seeds for reproducibility
random.seed(42)
torch.manual_seed(42)

# 1. Download Dataset
print("Downloading GTZAN Dataset...")
path = kagglehub.dataset_download(
    "andradaolteanu/gtzan-dataset-music-genre-classification"
)
source_dir = os.path.join(path, "Data", "genres_original")
target_root = "gtzan_audio_split"

# 2. Strict File-Level Split (80/10/10)
if not os.path.exists(target_root):
    print("Building strict isolated directories to prevent data leakage...")
    genres = [
        d for d in os.listdir(source_dir) if os.path.isdir(os.path.join(source_dir, d))
    ]

    for split in ["train", "val", "test"]:
        for genre in genres:
            os.makedirs(os.path.join(target_root, split, genre), exist_ok=True)

    for genre in genres:
        genre_path = os.path.join(source_dir, genre)
        files = [f for f in os.listdir(genre_path) if f.endswith(".wav")]

        # Remove known corrupted file
        if "jazz.00054.wav" in files:
            files.remove("jazz.00054.wav")

        random.shuffle(files)
        train_idx, val_idx = int(len(files) * 0.8), int(len(files) * 0.9)

        splits = {
            "train": files[:train_idx],
            "val": files[train_idx:val_idx],
            "test": files[val_idx:],
        }

        for split, split_files in splits.items():
            for f in split_files:
                shutil.copy(
                    os.path.join(genre_path, f),
                    os.path.join(target_root, split, genre, f),
                )
    print("Data successfully downloaded and split!")
else:
    print("Split directories already exist. Skipping split phase.")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Device: {device}")

In [ ]:
# ==========================================
# DATA PROCESSING & LOADERS
# ==========================================

class GTZAN_3Second_Dataset(Dataset):
    def __init__(self, root_dir, split, sample_rate=22050, duration=3, augment=False):
        self.root_dir = os.path.join(root_dir, split)
        self.sample_rate = sample_rate
        self.n_samples = sample_rate * duration
        self.augment = augment
        self.file_list = []

        genres = sorted(os.listdir(self.root_dir))
        self.label_to_idx = {genre: i for i, genre in enumerate(genres)}

        for genre in genres:
            genre_dir = os.path.join(self.root_dir, genre)
            for f in os.listdir(genre_dir):
                if f.endswith(".wav"):
                    full_path = os.path.join(genre_dir, f)
                    for segment_idx in range(10):
                        self.file_list.append(
                            (full_path, self.label_to_idx[genre], segment_idx)
                        )

        self.mel_spectrogram = torchaudio.transforms.MelSpectrogram(
            sample_rate=self.sample_rate, n_fft=2048, hop_length=512, n_mels=128
        )
        self.amplitude_to_db = torchaudio.transforms.AmplitudeToDB(top_db=80)
        self.pitch_shift = torchaudio.transforms.PitchShift(sample_rate, n_steps=2)

    def add_white_noise(self, waveform, noise_level=0.005):
        return waveform + (torch.randn_like(waveform) * noise_level)

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        path, label, segment_idx = self.file_list[idx]
        offset = segment_idx * self.n_samples

        try:
            waveform, sr = torchaudio.load(
                path, frame_offset=offset, num_frames=self.n_samples
            )
            if waveform.numel() == 0 or waveform.shape[1] == 0:
                waveform, sr = torch.zeros((1, self.n_samples)), self.sample_rate
        except Exception:
            waveform, sr = torch.zeros((1, self.n_samples)), self.sample_rate

        if sr != self.sample_rate:
            waveform = torchaudio.transforms.Resample(
                orig_freq=sr, new_freq=self.sample_rate
            )(waveform)
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)

        # 1. Augmentations FIRST
        if self.augment:
            if random.random() > 0.5:
                waveform = self.pitch_shift(waveform)
            if random.random() > 0.5:
                waveform = self.add_white_noise(waveform)

        # 2. Spectrogram Conversion
        mel_spec = self.mel_spectrogram(waveform)
        mel_spec_db = self.amplitude_to_db(mel_spec)

        # 3. Strict Padding/Truncating to guarantee 130 frames
        target_frames = 130
        if mel_spec_db.shape[2] < target_frames:
            mel_spec_db = F.pad(mel_spec_db, (0, target_frames - mel_spec_db.shape[2]))
        else:
            mel_spec_db = mel_spec_db[:, :, :target_frames]

        return mel_spec_db, torch.tensor(label)


# Initialize DataLoaders
batch_size = 32
train_loader = DataLoader(
    GTZAN_3Second_Dataset("gtzan_audio_split", "train", augment=True),
    batch_size=batch_size,
    shuffle=True,
)
val_loader = DataLoader(
    GTZAN_3Second_Dataset("gtzan_audio_split", "val", augment=False),
    batch_size=batch_size,
    shuffle=False,
)
test_loader = DataLoader(
    GTZAN_3Second_Dataset("gtzan_audio_split", "test", augment=False),
    batch_size=batch_size,
    shuffle=False,
)

# Print the exact size of the final datasets (in 3-second slices)
print(f"Total Training Samples: {len(train_loader.dataset)}")
print(f"Total Validation Samples: {len(val_loader.dataset)}")
print(f"Total Test Samples: {len(test_loader.dataset)}")

# Model Architecture

In [ ]:
# ==========================================
# MODEL ARCHITECTURES
# ==========================================

# --- 1. THE RHYTHM EXPERT (Custom Multi-Scale CNN) ---
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super(SEBlock, self).__init__()
        self.squeeze = nn.AdaptiveAvgPool2d(1)
        self.excitation = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.excitation(self.squeeze(x).view(b, c)).view(b, c, 1, 1)
        return x * y.expand_as(x)


class MultiScaleSpectroBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(MultiScaleSpectroBlock, self).__init__()
        self.conv_std = nn.Conv2d(
            in_channels, out_channels // 3, kernel_size=3, padding="same"
        )
        self.conv_tmp = nn.Conv2d(
            in_channels, out_channels // 3, kernel_size=(1, 7), padding="same"
        )
        self.conv_frq = nn.Conv2d(
            in_channels,
            out_channels - 2 * (out_channels // 3),
            kernel_size=(7, 1),
            padding="same",
        )
        self.bn = nn.BatchNorm2d(out_channels)
        self.se = SEBlock(out_channels)
        self.pool = nn.MaxPool2d(2, 2)

    def forward(self, x):
        out = torch.cat([self.conv_std(x), self.conv_tmp(x), self.conv_frq(x)], dim=1)
        return self.pool(self.se(F.leaky_relu(self.bn(out), 0.01)))


class SpectroCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(SpectroCNN, self).__init__()
        self.init_conv = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.01),
        )
        self.blocks = nn.Sequential(
            MultiScaleSpectroBlock(32, 64),
            MultiScaleSpectroBlock(64, 128),
            MultiScaleSpectroBlock(128, 256),
            MultiScaleSpectroBlock(256, 512),
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.LeakyReLU(0.01),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.pool(self.blocks(self.init_conv(x))).view(x.size(0), -1)
        return self.classifier(x)


# --- 2. THE TEXTURE EXPERT (ResNet-50 Transfer Learning) ---
class SpectroResNet50(nn.Module):
    def __init__(self, num_classes=10):
        super(SpectroResNet50, self).__init__()
        self.resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        # Collapse RGB to Grayscale safely
        orig_weight = self.resnet.conv1.weight.clone()
        self.resnet.conv1 = nn.Conv2d(
            1, 64, kernel_size=7, stride=2, padding=3, bias=False
        )
        with torch.no_grad():
            self.resnet.conv1.weight = nn.Parameter(
                torch.sum(orig_weight, dim=1, keepdim=True)
            )
        # Custom head
        num_ftrs = self.resnet.fc.in_features
        self.resnet.fc = nn.Sequential(
            nn.Dropout(0.5), nn.Linear(num_ftrs, num_classes)
        )

    def forward(self, x):
        return self.resnet(x)


# --- 3. THE ULTIMATE ENSEMBLE ---
class UltimateEnsemble(nn.Module):
    def __init__(self, model_a, model_b):
        super(UltimateEnsemble, self).__init__()
        self.model_a = model_a
        self.model_b = model_b

    def forward(self, x):
        prob_a = F.softmax(self.model_a(x), dim=1)
        prob_b = F.softmax(self.model_b(x), dim=1)
        return (prob_a + prob_b) / 2.0

In [ ]:
# ==========================================
# TRAINING LOOP
# ==========================================

def train_model(model, model_name, epochs, lr, train_loader, val_loader, device):
    print(f"\n{'='*50}\n STARTING TRAINING: {model_name}\n{'='*50}")
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs)

    best_val_acc = 0.0
    start_time = time.time()

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            running_loss += loss.item()

        avg_train_loss = running_loss / len(train_loader)

        # Validation
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        val_acc = 100 * correct / total
        scheduler.step()

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), f"best_{model_name}.pth")
            msg = "--> Best Model Saved!"
        else:
            msg = ""

        print(
            f"Epoch [{epoch+1:02d}/{epochs}] | Train Loss: {avg_train_loss:.4f} | Val Acc: {val_acc:.2f}% {msg}"
        )

    print(
        f"\n[{model_name}] Training Completed in {(time.time() - start_time) / 60:.2f} minutes!"
    )
    print(f"[{model_name}] Absolute Best Validation Accuracy: {best_val_acc:.2f}%")
    return model


# 1. Train Custom CNN
model_cnn = SpectroCNN(num_classes=10).to(device)
train_model(
    model_cnn,
    "spectro_cnn",
    epochs=40,
    lr=0.001,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
)

# 2. Fine-Tune ResNet-50
model_resnet = SpectroResNet50(num_classes=10).to(device)
train_model(
    model_resnet,
    "resnet50",
    epochs=20,
    lr=0.0005,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
)

In [ ]:
# ==========================================
# EVALUATION
# ==========================================

print("Loading optimized weights into the Ensemble...")
expert_cnn = SpectroCNN(num_classes=10).to(device)
expert_cnn.load_state_dict(torch.load("best_spectro_cnn.pth"))
expert_cnn.eval()

expert_resnet = SpectroResNet50(num_classes=10).to(device)
expert_resnet.load_state_dict(torch.load("best_resnet50.pth"))
expert_resnet.eval()

ultimate_model = UltimateEnsemble(expert_cnn, expert_resnet).to(device)
ultimate_model.eval()

y_true = []
y_pred = []

print("Running Inference on Isolated Test Set...")
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = ultimate_model(inputs)
        _, predicted = torch.max(outputs, 1)
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())

genres = sorted(os.listdir("gtzan_audio_split/test"))

print("\n" + "=" * 50)
print(" ULTIMATE ENSEMBLE CLASSIFICATION REPORT")
print("=" * 50)
print(classification_report(y_true, y_pred, target_names=genres))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(12, 9))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="viridis",
    xticklabels=genres,
    yticklabels=genres,
    linewidths=0.5,
    cbar_kws={"shrink": 0.75},
)

plt.title("Ultimate Ensemble (CNN + ResNet50) - Confusion Matrix", fontsize=16, pad=20)
plt.ylabel("Actual True Genre", fontsize=12, labelpad=10)
plt.xlabel("Model Predicted Genre", fontsize=12, labelpad=10)
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()